# Hands-on 7: Membersihkan Timestamp, Resampling, dan Imputasi Sederhana

Buku Referensi: *IoT Terintegrasi dengan Machine Learning & Data Analytics* (Onno W. Purbo), Bab 4.

Alur: parse timestamp -> buang invalid -> urutkan -> hapus duplikat -> resample 10 menit ->
imputasi (forward fill vs interpolasi linier) -> simpan `data/processed/room_sensor_clean.csv`.

In [1]:
import os
from pathlib import Path

while not (Path.cwd() / "scripts" / "mqtt").is_dir():
    os.chdir("..")
print("[INFO] Working directory:", Path.cwd())

[INFO] Working directory: /home/sandbox/iot-ml-book


## 1. Bangkitkan Dataset Kotor (masalah khas IoT)

In [2]:
import subprocess, sys
r = subprocess.run([sys.executable, "scripts/generate_dirty_dataset.py"],
                   capture_output=True, text=True)
print(r.stdout)

[SUKSES] Dataset kotor tersimpan: data/raw/room_sensor_dirty.csv (1003 baris)
[INFO] Masalah disuntikkan - invalid_timestamp: 20
[INFO] Masalah disuntikkan - duplikat: 15
[INFO] Masalah disuntikkan - nilai_hilang: 120
[INFO] Masalah disuntikkan - baris_gap: 20



## 2. Cleaning, Resampling 10 Menit, dan Imputasi

In [3]:
r = subprocess.run([sys.executable, "scripts/clean_timestamp_resample.py"],
                   capture_output=True, text=True)
print(r.stdout)

=== Data awal ===
             timestamp device_id  ...  co2_ppm  power_watt
0  2026-01-03 05:40:00    room01  ...    453.0       87.60
1  2026-01-01 15:50:00    room01  ...    773.0      140.14
2  2026-01-06 20:30:00    room01  ...    430.0       88.67
3          bukan-waktu    room01  ...    887.0      160.93
4  2026-01-05 19:20:00    room01  ...    450.0       87.96

[5 rows x 6 columns]
Jumlah baris awal: 1003

=== Setelah membersihkan timestamp ===
              timestamp device_id  ...  co2_ppm  power_watt
564 2026-01-01 00:00:00    room01  ...    431.0       85.57
785 2026-01-01 00:10:00    room01  ...    457.0       83.39
360 2026-01-01 00:20:00    room01  ...    431.0       90.09
427 2026-01-01 00:30:00    room01  ...    437.0       90.72
435 2026-01-01 00:40:00    room01  ...    453.0       82.10

[5 rows x 6 columns]
Jumlah baris setelah clean timestamp: 968



## 3. Perbandingan Visual Sebelum/Sesudah

In [4]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import pandas as pd

df_dirty = pd.read_csv("data/raw/room_sensor_dirty.csv")
df_dirty["timestamp"] = pd.to_datetime(df_dirty["timestamp"], errors="coerce")
df_dirty = df_dirty.dropna(subset=["timestamp"]).sort_values("timestamp")
df_clean = pd.read_csv("data/processed/room_sensor_clean.csv", parse_dates=["timestamp"])

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)
axes[0].plot(df_dirty["timestamp"], df_dirty["temperature_c"], ".", ms=2, color="tomato")
axes[0].set_title("Sebelum Cleaning - Suhu (timestamp kotor, duplikat, gap, NaN)")
axes[0].set_ylabel("Suhu (C)")
axes[1].plot(df_clean["timestamp"], df_clean["temperature_c"], "-", lw=1, color="seagreen")
axes[1].set_title("Sesudah Cleaning - Resample 10 menit + Interpolasi Linier")
axes[1].set_ylabel("Suhu (C)")
axes[1].set_xlabel("Waktu")
plt.xticks(rotation=45)
plt.tight_layout()
Path("outputs/figures").mkdir(parents=True, exist_ok=True)
plt.savefig("outputs/figures/cleaning_before_after.png")
plt.show()
print("[SUKSES] outputs/figures/cleaning_before_after.png")

[SUKSES] outputs/figures/cleaning_before_after.png


## Penutup

Data kotor (timestamp invalid, duplikat, nilai hilang, gap interval) berhasil dibersihkan
menjadi deret 10 menit yang konsisten tanpa nilai hilang. Dataset bersih ini menjadi dasar
deteksi anomali statistik pada Hands-on 8.

## Catatan dari buku: pengamatan, variasi, dan latihan

Bagian berikut disalin dari buku (Hands-on 7 dalam urutan buku) sebagai panduan pengamatan dan latihan mandiri setelah menjalankan notebook ini.

### 10. Langkah kerja yang harus dilakukan pembaca

Tahap

Kegiatan

Baca file kotor.

Parse timestamp.

Buang waktu invalid.

Urutkan data dan hapus duplikasi.

Set index waktu.

Lakukan resample ke interval target.

Lakukan imputasi sederhana.

Simpan hasil ke folder processed.